## InSAR Science 

In [ ]:
#import module section







In [ ]:
from pathlib import Path
import sys

print("Python executable:", sys.executable)
print("Current folder:", Path.cwd())


In [ ]:
# =============================================================================
# InSAR-Sci official configuration
# =============================================================================
# This notebook works on exported GeoTIFF products, not raw GAMMA binary files.
# Keep code in the Git repo. Keep data and outputs outside Git.

from pathlib import Path
import numpy as np


CONFIG = {
    # =========================================================================
    # Project identity
    # =========================================================================
    "project": {
        "name": "InSAR_Sci",
        "analysis_version": "A15_dev",
        "track_id": "A166",
        "orbit_geometry": "ascending",
        "description": (
            "GeoTIFF-based post-processing, visualization, comparison, and "
            "candidate SSE diagnostics for GAMMA-derived Sentinel-1 DInSAR products."
        ),
    },

    # =========================================================================
    # Input/output paths
    # =========================================================================
    "paths": {
        "geotiff_root": Path("/mnt/raid5/fort_john_InSAR/GroupA/GeoBase"),
        "manifest_csv": Path("/mnt/raid5/fort_john_InSAR/2024_2025_SSE_15.csv"),
        "dem_path": Path("/mnt/raid5/fort_john_InSAR/GroupA/inputs/SRTM1.tif"),
        "sse_points": Path(
            "/mnt/raid5/fort_john_InSAR/GIS/SSE_Candidates/SSE_Candidates.shp"
        ),
        "output_root": Path("/mnt/raid5/fort_john_InSAR/InSAR_Sci_A15"),
    },

    # =========================================================================
    # Output organization
    # =========================================================================
    "outputs": {
        "tables_subdir": "tables",
        "quicklooks_subdir": "quicklooks",
        "master_groups_subdir": "master_groups",
        "comparison_subdir": "comparisons",
        "dem_diagnostics_subdir": "dem_diagnostics",
        "logs_subdir": "logs",

        "sse_root_subdir": "candi_sse_geo",
        "sse_plots_subdir": "candi_sse_geo/plots",
        "sse_rasters_subdir": "candi_sse_geo/geotiffs",
        "sse_tables_subdir": "candi_sse_geo/tables",
    },

    # =========================================================================
    # Manifest conventions
    # =========================================================================
    "manifest": {
        "pair_id_column": "pair_id",

        "boolean_columns": [
            "SSE",
            "Noise",
            "Dirty",
            "processing issues",
            "Interesting Feature",
            "Plane Removal",
            "GACOS",
            "Use GACOS",
            "export_ok",
            "plot_ok",
        ],

        # Future column. The code tolerates it being absent.
        "future_boolean_columns": [
            "Use atm correction",
        ],

        "sse_id_columns": ["SSE ID", "SSE2 ID"],
    },

    # =========================================================================
    # Base GeoTIFF product naming
    # =========================================================================
    "products": {
        "geo_subdir": "geo",

        "suffixes": {
            # Current products.
            "disp": "_disp.tif",
            "disp_pr": "_disp_pr.tif",
            "coh": "_coh.tif",
            "coh_raw": "_coh_raw.tif",
            "phase_raw": "_phase_raw.tif",
            "phase_filt": "_phase_filt.tif",
            "unw": "_unw.tif",
            "unw_pr": "_unw_pr.tif",

            # Future placeholders. Update once real corrected files exist.
            "disp_gacos": "_disp_gacos.tif",
            "disp_pr_gacos": "_disp_pr_gacos.tif",
            "disp_atm": "_disp_atm.tif",
            "disp_pr_gacos_atm": "_disp_pr_gacos_atm.tif",

            "unw_gacos": "_unw_gacos.tif",
            "unw_pr_gacos": "_unw_pr_gacos.tif",
            "unw_atm": "_unw_atm.tif",
            "unw_pr_gacos_atm": "_unw_pr_gacos_atm.tif",

            "phase_filt_gacos": "_phase_filt_gacos.tif",
            "phase_filt_atm": "_phase_filt_atm.tif",
            "phase_filt_gacos_atm": "_phase_filt_gacos_atm.tif",
        },

        "default_plot_products": ["disp", "coh"],

        "optional_plot_products": [
            "disp_pr",
            "coh_raw",
            "phase_raw",
            "phase_filt",
            "unw",
            "unw_pr",
        ],

        "classes": {
            "displacement": [
                "disp",
                "disp_pr",
                "disp_gacos",
                "disp_pr_gacos",
                "disp_atm",
                "disp_pr_gacos_atm",
            ],
            "coherence": [
                "coh",
                "coh_raw",
            ],
            "wrapped_phase": [
                "phase_raw",
                "phase_filt",
                "phase_filt_gacos",
                "phase_filt_atm",
                "phase_filt_gacos_atm",
            ],
            "unwrapped_phase": [
                "unw",
                "unw_pr",
                "unw_gacos",
                "unw_pr_gacos",
                "unw_atm",
                "unw_pr_gacos_atm",
            ],
        },
    },

    # =========================================================================
    # Processing/correction variants
    # =========================================================================
    "variants": {
        "raw": {
            "active": True,
            "label": "Raw",
            "description": (
                "Original exported products without additional post-export correction."
            ),
            "manifest_use_column": None,
        },

        "plane_removed": {
            "active": True,
            "label": "Plane removed",
            "description": (
                "Products after spatial plane/ramp removal. Useful for visualization "
                "and local diagnostics, but may remove real long-wavelength deformation."
            ),
            "manifest_use_column": "Plane Removal",
        },

        "gacos": {
            "active": False,
            "label": "GACOS corrected",
            "description": (
                "Future GACOS-corrected products. Activate only when files exist and "
                "the manifest column indicates the correction should be used."
            ),
            "manifest_use_column": "Use GACOS",
        },

        "plane_removed_gacos": {
            "active": False,
            "label": "Plane removed + GACOS",
            "description": (
                "Future products combining plane/ramp removal and GACOS correction. "
                "Use carefully because both operations can alter long-wavelength patterns."
            ),
            "manifest_use_column": "Use GACOS",
        },

        "atm_corrected": {
            "active": False,
            "label": "Atmospheric corrected",
            "description": (
                "Future products corrected using data-driven atmospheric phase estimates, "
                "for example common-scene/common-reference approaches."
            ),
            "manifest_use_column": "Use atm correction",
        },

        "plane_removed_gacos_atm": {
            "active": False,
            "label": "Plane removed + GACOS + atmospheric corrected",
            "description": (
                "Future fully corrected variant. Use only when processing provenance is "
                "explicit in the manifest and filenames."
            ),
            "manifest_use_column": "Use atm correction",
        },
    },

    # =========================================================================
    # Product selection policy
    # =========================================================================
    "product_selection": {
        # Allowed: "auto_from_manifest", "explicit_variant"
        "mode": "auto_from_manifest",

        # Used only when mode == "explicit_variant".
        "selected_variant": "raw",

        "manifest_columns": {
            "plane_removed": "Plane Removal",
            "gacos": "Use GACOS",
            "atm_corrected": "Use atm correction",
        },

        "missing_flag_policy": "raw",

        # product_role -> actual product_key
        "role_product_maps": {
            "raw": {
                "disp": "disp",
                "coh": "coh",
                "coh_raw": "coh_raw",
                "phase": "phase_filt",
                "phase_raw": "phase_raw",
                "unw": "unw",
            },

            "plane_removed": {
                "disp": "disp_pr",
                "coh": "coh",
                "coh_raw": "coh_raw",
                "phase": "phase_filt",
                "phase_raw": "phase_raw",
                "unw": "unw_pr",
            },

            "gacos": {
                "disp": "disp_gacos",
                "coh": "coh",
                "phase": "phase_filt_gacos",
                "unw": "unw_gacos",
            },

            "plane_removed_gacos": {
                "disp": "disp_pr_gacos",
                "coh": "coh",
                "phase": "phase_filt_gacos",
                "unw": "unw_pr_gacos",
            },

            "atm_corrected": {
                "disp": "disp_atm",
                "coh": "coh",
                "phase": "phase_filt_atm",
                "unw": "unw_atm",
            },

            "plane_removed_gacos_atm": {
                "disp": "disp_pr_gacos_atm",
                "coh": "coh",
                "phase": "phase_filt_gacos_atm",
                "unw": "unw_pr_gacos_atm",
            },
        },
    },

    # =========================================================================
    # Masking policy
    # =========================================================================
    "masking": {
        # Respect file nodata to remove exported zero-padding outside the footprint.
        # This is not the same as declaring every physical zero invalid.
        "respect_file_nodata_by_product": {
            "disp": True,
            "disp_pr": True,
            "coh": True,
            "coh_raw": True,
            "phase_raw": True,
            "phase_filt": True,
            "unw": True,
            "unw_pr": True,

            "disp_gacos": True,
            "disp_pr_gacos": True,
            "disp_atm": True,
            "disp_pr_gacos_atm": True,
            "unw_gacos": True,
            "unw_pr_gacos": True,
            "unw_atm": True,
            "unw_pr_gacos_atm": True,
            "phase_filt_gacos": True,
            "phase_filt_atm": True,
            "phase_filt_gacos_atm": True,
        },

        # Additional zero masking beyond file nodata. Keep False unless proven needed.
        "zero_is_nodata_by_product": {
            "disp": False,
            "disp_pr": False,
            "coh": False,
            "coh_raw": False,
            "phase_raw": False,
            "phase_filt": False,
            "unw": False,
            "unw_pr": False,

            "disp_gacos": False,
            "disp_pr_gacos": False,
            "disp_atm": False,
            "disp_pr_gacos_atm": False,
            "unw_gacos": False,
            "unw_pr_gacos": False,
            "unw_atm": False,
            "unw_pr_gacos_atm": False,
            "phase_filt_gacos": False,
            "phase_filt_atm": False,
            "phase_filt_gacos_atm": False,
        },

        # Optional coherence masking for later analysis. Keep off until tested.
        "use_coherence_mask": False,
        "coherence_product": "coh",
        "coherence_threshold": 0.4,

        # Used for diagnostics/plots, not for modifying rasters.
        "valid_percentile_clip": (2.0, 98.0),

        # For display/analysis support of derived products.
        # Example: disp_pr should not show pixels outside the raw disp footprint.
        "plot_footprint_mask": {
            "enabled": True,

            "reference_product_by_role": {
                "disp": "disp",
                "unw": "disp",
                "phase": "disp",
                "coh": "coh",
            },

            # Reprojection belongs in comparison.py later, not plotting.py.
            "on_grid_mismatch": "warn",
        },
    },

    # =========================================================================
    # Plot defaults
    # =========================================================================
    "plotting": {
        "show_plots": True,
        "save_plots": False,
        "dpi": 300,

        # Display only. Statistics use full-resolution arrays.
        "plot_decimation": 6,
        "figsize_single": (8, 7),
        "figsize_per_panel": (7, 5.5),

        "title_detail_mode": "footnote",  # allowed: "full", "footnote"
        "footnote_fontsize": 8,

        "colormaps": {
            "disp": "RdBu_r",
            "disp_pr": "RdBu_r",
            "disp_gacos": "RdBu_r",
            "disp_pr_gacos": "RdBu_r",
            "disp_atm": "RdBu_r",
            "disp_pr_gacos_atm": "RdBu_r",

            "unw": "turbo",
            "unw_pr": "turbo",
            "unw_gacos": "turbo",
            "unw_pr_gacos": "turbo",
            "unw_atm": "turbo",
            "unw_pr_gacos_atm": "turbo",

            "phase_raw": "turbo",
            "phase_filt": "turbo",
            "phase_filt_gacos": "turbo",
            "phase_filt_atm": "turbo",
            "phase_filt_gacos_atm": "turbo",

            "coh": "gray",
            "coh_raw": "gray",
        },

        "units": {
            "disp": "LOS displacement [m]",
            "disp_pr": "LOS displacement, plane-removed [m]",
            "disp_gacos": "LOS displacement, GACOS-corrected [m]",
            "disp_pr_gacos": "LOS displacement, plane-removed + GACOS [m]",
            "disp_atm": "LOS displacement, atmospheric-corrected [m]",
            "disp_pr_gacos_atm": (
                "LOS displacement, plane-removed + GACOS + atm-corrected [m]"
            ),

            "unw": "Unwrapped phase [rad]",
            "unw_pr": "Unwrapped phase, plane-removed [rad]",
            "unw_gacos": "Unwrapped phase, GACOS-corrected [rad]",
            "unw_pr_gacos": "Unwrapped phase, plane-removed + GACOS [rad]",
            "unw_atm": "Unwrapped phase, atmospheric-corrected [rad]",
            "unw_pr_gacos_atm": (
                "Unwrapped phase, plane-removed + GACOS + atm-corrected [rad]"
            ),

            "phase_raw": "Wrapped phase [rad]",
            "phase_filt": "Filtered wrapped phase [rad]",
            "phase_filt_gacos": "Filtered wrapped phase, GACOS-corrected [rad]",
            "phase_filt_atm": "Filtered wrapped phase, atmospheric-corrected [rad]",
            "phase_filt_gacos_atm": "Filtered wrapped phase, GACOS + atm-corrected [rad]",

            "coh": "Coherence",
            "coh_raw": "Raw coherence",
        },

        "robust_percentile": 98.0,
        "symmetric_displacement_limits": True,
        "wrapped_phase_limits": (-np.pi, np.pi),
        "coherence_limits": (0.0, 1.0),

        # Visualization-only normalization.
        "display_normalization": {
            "enabled": True,

            "method_by_product": {
                "disp": "mean",
                "disp_pr": "mean",
                "disp_gacos": "mean",
                "disp_pr_gacos": "mean",
                "disp_atm": "mean",
                "disp_pr_gacos_atm": "mean",

                "unw": "mean",
                "unw_pr": "mean",
                "unw_gacos": "mean",
                "unw_pr_gacos": "mean",
                "unw_atm": "mean",
                "unw_pr_gacos_atm": "mean",

                "coh": "none",
                "coh_raw": "none",
                "phase_raw": "none",
                "phase_filt": "none",
                "phase_filt_gacos": "none",
                "phase_filt_atm": "none",
                "phase_filt_gacos_atm": "none",
            },

            "title_suffix": True,
            "show_raw_mean_median_in_title": True,
        },
    },

    # =========================================================================
    # Pair/master comparison settings
    # =========================================================================
    "comparison": {
        "reference_grid": "first_pair",

        "metrics": [
            "rmse",
            "mae",
            "bias",
            "median_abs",
            "ks",
            "wasserstein",
            "pearson",
            "spearman",
        ],

        "cdf": {
            "enabled": True,
            "n_points": 1000,
            "plot_ks_gap": True,
        },

        "histogram": {
            "enabled": True,
            "bins": 100,
            "density": True,
        },

        "resampling": {
            "continuous": "bilinear",
            "categorical": "nearest",
        },

        # Official metrics should stay raw unless explicitly testing normalization.
        "metric_normalization": {
            "default": "none",
            "allow_alternative_modes": ["none", "mean", "median"],
        },

        "default_product_role": "disp",
        "default_variants_to_compare": ["raw", "plane_removed"],
        "respect_manifest_use_columns": True,
    },

    # =========================================================================
    # Candidate SSE local AOI analysis
    # =========================================================================
    "sse_aoi": {
        "id_column": "id",
        "box_size_km": 12.0,

        "require_sse_candidate": True,
        "require_sse_id": True,

        "crop_product_roles": ["disp", "coh", "phase", "unw"],
        "default_variant": "auto_from_manifest",

        "optional_variants": [
            "raw",
            "plane_removed",
            "gacos",
            "plane_removed_gacos",
            "atm_corrected",
        ],

        "local_stats": [
            "mean",
            "median",
            "std",
            "min",
            "max",
            "p05",
            "p50",
            "p95",
            "positive_area_fraction",
            "negative_area_fraction",
        ],
    },

    # =========================================================================
    # DEM/topographic diagnostics
    # =========================================================================
    "dem_diagnostics": {
        "enabled": True,

        # Diagnostic only. Displacement/phase vs DEM can indicate atmospheric
        # delay or residual topo error, not physical elevation gain/loss.
        "product_roles": ["disp", "unw"],
        "variants": ["raw", "plane_removed"],

        "scatter": {
            "enabled": True,
            "sample_max_pixels": 300_000,
            "alpha": 0.15,
            "point_size": 1,
        },

        "fit_models": {
            "linear_phase_height": True,
            "linear_disp_height": True,

            # Future only.
            "exponential_height_model": False,
            "h0_m": 7000.0,
            "alpha": 1.35,
        },

        "save_tables": True,
        "save_plots": True,
    },

    # =========================================================================
    # Future lobe/shape dynamics diagnostics
    # =========================================================================
    "lobe_diagnostics": {
        "enabled": False,

        # Do not interpret as SSE propagation until atmospheric/reference effects
        # are controlled.
        "product_role": "disp",
        "variants": ["raw", "plane_removed"],

        "threshold_method": "robust_sigma",
        "sigma_threshold": 2.0,

        "metrics": [
            "positive_centroid",
            "negative_centroid",
            "lobe_separation_km",
            "orientation_deg",
            "positive_area_km2",
            "negative_area_km2",
            "positive_peak",
            "negative_peak",
            "cross_correlation_peak",
        ],
    },

    # =========================================================================
    # Run control
    # =========================================================================
    "run_control": {
        "use_first_n_rows": 15,
        "dry_run": False,
        "verbose": True,
        "fail_on_missing_required_products": True,
    },
}

In [ ]:
print("CONFIG OK")
print("analysis_version:", CONFIG["project"]["analysis_version"])
print("product_selection mode:", CONFIG["product_selection"]["mode"])
print("footprint mask enabled:", CONFIG["masking"]["plot_footprint_mask"]["enabled"])
print("title mode:", CONFIG["plotting"]["title_detail_mode"])
print("figsize_per_panel:", CONFIG["plotting"]["figsize_per_panel"])

In [ ]:
# 1. Imports
# 2. CONFIG
# 3. Official run setup:
#    - create output folders
#    - load manifest
#    - prepare manifest
#    - apply development subset
#    - build GeoTIFF inventory
#    - save manifest summary table
#    - save product inventory table
# 4. Raster metadata summary
# 5. First quicklook plots
# 6. Master-group comparison plots
# 7. SSE AOI extraction
# 8. Local comparison metrics
# 9. DEM/phase/displacement diagnostics
# 10. Summary figures and tables

In [ ]:
# persistence inspection, not proof of SSE propagation

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import rasterio

# ---------------------------------------------------------------------
# Basic path checks
# ---------------------------------------------------------------------
paths = CONFIG["paths"]

for key, path in paths.items():
    path = Path(path)
    print(f"{key:15s}: {path}")
    print(f"{'':15s}  exists = {path.exists()}")

# ---------------------------------------------------------------------
# Manifest inspection
# ---------------------------------------------------------------------
manifest = pd.read_csv(paths["manifest_csv"])

print("\n--- Manifest ---")
print("Shape:", manifest.shape)
print("Columns:")
for col in manifest.columns:
    print("  -", col)

print("\nFirst rows:")
display(manifest.head(CONFIG["run_control"]["use_first_n_rows"]))

print("\nMissing values per column:")
display(manifest.isna().sum().sort_values(ascending=False))

# ---------------------------------------------------------------------
# Pair/product inventory for first N rows
# ---------------------------------------------------------------------
suffixes = CONFIG["products"]["suffixes"]
geo_root = Path(paths["geotiff_root"])
geo_subdir = CONFIG["products"]["geo_subdir"]

n = CONFIG["run_control"]["use_first_n_rows"]
dev_manifest = manifest.head(n).copy()

print("\n--- Product existence check ---")
records = []

for _, row in dev_manifest.iterrows():
    pair_id = str(row["pair_id"])
    pair_geo_dir = geo_root / pair_id / geo_subdir

    rec = {
        "pair_id": pair_id,
        "pair_geo_dir_exists": pair_geo_dir.exists(),
    }

    for product_key, suffix in suffixes.items():
        tif_path = pair_geo_dir / f"{pair_id}{suffix}"
        rec[product_key] = tif_path.exists()

    records.append(rec)

inventory = pd.DataFrame(records)
display(inventory)

print("\nMissing products:")
missing = inventory.set_index("pair_id").drop(columns=["pair_geo_dir_exists"]) == False
display(missing.loc[missing.any(axis=1)])

# ---------------------------------------------------------------------
# Inspect one representative raster
# ---------------------------------------------------------------------
first_pair = str(dev_manifest.iloc[0]["pair_id"])
first_disp = geo_root / first_pair / geo_subdir / f"{first_pair}_disp.tif"

print("\n--- Representative raster metadata ---")
print(first_disp)

with rasterio.open(first_disp) as src:
    print("CRS:", src.crs)
    print("Shape:", (src.height, src.width))
    print("Bounds:", src.bounds)
    print("Transform:", src.transform)
    print("Resolution:", src.res)
    print("Nodata:", src.nodata)
    print("Dtype:", src.dtypes)
    arr = src.read(1, masked=True)
    print("Masked array:", type(arr), arr.shape)
    print("Valid count:", arr.count())
    print("Min / Max:", float(arr.min()), float(arr.max()))

# ---------------------------------------------------------------------
# SSE shapefile inspection
# ---------------------------------------------------------------------
print("\n--- SSE candidate points ---")
sse_gdf = gpd.read_file(paths["sse_points"])
print("CRS:", sse_gdf.crs)
print("Columns:", list(sse_gdf.columns))
print("Geometry types:", sse_gdf.geometry.geom_type.unique())
display(sse_gdf.head())

In [ ]:
%%writefile /home/camilo/Documents/InSAR_Science/src/insar_sci/plotting.py
"""
Plotting utilities for InSAR-Sci.

Main idea:
- product_role = scientific request: disp, coh, phase, unw
- product_key  = actual file: disp, disp_pr, coh, phase_filt, unw_pr

In auto mode:
    Plane Removal=True  -> disp role uses disp_pr
    Plane Removal=False -> disp role uses disp

Display normalization is only for plotting. Raw raster values are not modified.
"""

from __future__ import annotations

from pathlib import Path
from typing import Iterable, Mapping

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from insar_sci.paths import (
    product_key_for_role,
    product_path,
    product_path_for_role,
    resolve_variant_from_row,
)
from insar_sci.raster_io import read_product_array, raster_stats, robust_limits


def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def decimate_for_plot(array: np.ndarray, decimation: int = 1) -> np.ndarray:
    if decimation is None or decimation <= 1:
        return array
    return array[::decimation, ::decimation]


def product_class(product_key: str, config: dict) -> str:
    classes = config["products"].get("classes", {})

    for class_name, product_keys in classes.items():
        if product_key in product_keys:
            return class_name

    return "other"


def infer_product_role_from_key(product_key: str, config: dict) -> str | None:
    pclass = product_class(product_key, config)

    if pclass == "displacement":
        return "disp"
    if pclass == "coherence":
        return "coh"
    if pclass == "wrapped_phase":
        return "phase"
    if pclass == "unwrapped_phase":
        return "unw"

    return None


def same_grid(metadata_a, metadata_b) -> bool:
    return (
        metadata_a.shape == metadata_b.shape
        and metadata_a.crs == metadata_b.crs
        and metadata_a.transform == metadata_b.transform
    )


def format_axis(ax) -> None:
    ax.set_xlabel("Longitude [°]")
    ax.set_ylabel("Latitude [°]")
    ax.ticklabel_format(useOffset=False)


def default_footprint_config(config: dict) -> dict:
    """
    Return footprint-mask config.

    If CONFIG does not yet define masking.plot_footprint_mask, this gives
    the desired default behavior: use raw disp as the footprint for derived
    displacement/phase/unwrapped products.
    """
    default = {
        "enabled": True,
        "reference_product_by_role": {
            "disp": "disp",
            "unw": "disp",
            "phase": "disp",
            "coh": "coh",
        },
        "on_grid_mismatch": "warn",
    }

    user_cfg = config.get("masking", {}).get("plot_footprint_mask", {})

    out = default.copy()
    out.update(user_cfg)

    if "reference_product_by_role" in user_cfg:
        ref = default["reference_product_by_role"].copy()
        ref.update(user_cfg["reference_product_by_role"])
        out["reference_product_by_role"] = ref

    return out


def apply_display_normalization(
    array: np.ndarray,
    product_key: str,
    config: dict,
) -> tuple[np.ndarray, str, float | None]:
    normalization = config["plotting"].get("display_normalization", {})

    if not normalization.get("enabled", False):
        return array, "raw", None

    method = normalization.get("method_by_product", {}).get(product_key, "none")

    if method == "none":
        return array, "raw", None

    values = array[np.isfinite(array)]

    if values.size == 0:
        return array, "raw", None

    if method == "mean":
        offset = float(np.nanmean(values))
        return array - offset, f"mean removed ({offset:.5g})", offset

    if method == "median":
        offset = float(np.nanmedian(values))
        return array - offset, f"median removed ({offset:.5g})", offset

    raise ValueError(
        f"Unknown display normalization method {method!r}. "
        "Use 'none', 'mean', or 'median'."
    )


def product_plot_spec(
    product_key: str,
    array: np.ndarray,
    config: dict,
    shared_limits: tuple[float, float] | None = None,
) -> dict[str, object]:
    plotting = config["plotting"]
    pclass = product_class(product_key, config)

    cmap = plotting["colormaps"].get(product_key, "viridis")
    label = plotting["units"].get(product_key, product_key)

    if shared_limits is not None:
        vmin, vmax = shared_limits

    elif pclass == "coherence":
        vmin, vmax = plotting.get("coherence_limits", (0.0, 1.0))

    elif pclass == "wrapped_phase":
        vmin, vmax = plotting.get("wrapped_phase_limits", (-np.pi, np.pi))

    elif pclass == "displacement":
        vmin, vmax = robust_limits(
            array,
            percentile=plotting.get("robust_percentile", 98.0),
            symmetric=plotting.get("symmetric_displacement_limits", True),
        )

    else:
        vmin, vmax = robust_limits(
            array,
            percentile=plotting.get("robust_percentile", 98.0),
            symmetric=False,
        )

    return {
        "cmap": cmap,
        "label": label,
        "vmin": vmin,
        "vmax": vmax,
    }


def shared_limits_for_items(
    items: list[dict[str, object]],
    config: dict,
) -> tuple[float, float]:
    product_keys = [item["product_key"] for item in items]
    product_classes = {product_class(key, config) for key in product_keys}

    if product_classes == {"coherence"}:
        return config["plotting"].get("coherence_limits", (0.0, 1.0))

    if product_classes == {"wrapped_phase"}:
        return config["plotting"].get("wrapped_phase_limits", (-np.pi, np.pi))

    symmetric = product_classes == {"displacement"}
    limits = []

    for item in items:
        array = item["display_array"]
        vmin, vmax = robust_limits(
            array,
            percentile=config["plotting"].get("robust_percentile", 98.0),
            symmetric=symmetric,
        )

        if np.isfinite(vmin) and np.isfinite(vmax):
            limits.append((vmin, vmax))

    if not limits:
        return float("nan"), float("nan")

    if symmetric:
        vmax_abs = max(max(abs(vmin), abs(vmax)) for vmin, vmax in limits)
        return -vmax_abs, vmax_abs

    return min(vmin for vmin, _ in limits), max(vmax for _, vmax in limits)


def read_product_for_plot(
    pair_id: str,
    config: dict,
    product_key: str | None = None,
    product_role: str | None = None,
    row: pd.Series | Mapping[str, object] | None = None,
    variant: str = "auto",
) -> dict[str, object]:
    if product_key is None and product_role is None:
        raise ValueError("Provide either product_key or product_role.")

    if product_key is not None and product_role is not None:
        raise ValueError("Provide only one of product_key or product_role.")

    if product_role is not None:
        resolved_variant = resolve_variant_from_row(
            row=row,
            config=config,
            variant=variant,
        )

        product_key = product_key_for_role(
            product_role=product_role,
            config=config,
            row=row,
            variant=variant,
        )

        path = product_path_for_role(
            pair_id=pair_id,
            product_role=product_role,
            config=config,
            row=row,
            variant=variant,
        )

        footprint_role = product_role

    else:
        resolved_variant = "explicit_product_key"

        path = product_path(
            geotiff_root=config["paths"]["geotiff_root"],
            pair_id=pair_id,
            product_key=product_key,
            suffixes=config["products"]["suffixes"],
            geo_subdir=config["products"]["geo_subdir"],
        )

        footprint_role = infer_product_role_from_key(product_key, config)

    raw_array, metadata, selected_mask, raw_stats_before = read_product_array(
        path=path,
        product_key=product_key,
        config=config,
    )

    array_after_footprint = raw_array.copy()

    footprint_cfg = default_footprint_config(config)
    footprint_applied = False
    footprint_product_key = None
    footprint_warning = None

    if footprint_cfg.get("enabled", False) and footprint_role is not None:
        footprint_product_key = footprint_cfg["reference_product_by_role"].get(
            footprint_role,
            None,
        )

        if footprint_product_key is not None:
            footprint_path = product_path(
                geotiff_root=config["paths"]["geotiff_root"],
                pair_id=pair_id,
                product_key=footprint_product_key,
                suffixes=config["products"]["suffixes"],
                geo_subdir=config["products"]["geo_subdir"],
            )

            _, footprint_metadata, footprint_mask, _ = read_product_array(
                path=footprint_path,
                product_key=footprint_product_key,
                config=config,
            )

            if same_grid(metadata, footprint_metadata):
                array_after_footprint[~footprint_mask] = np.nan
                footprint_applied = True
            else:
                footprint_warning = (
                    f"{product_key} grid differs from {footprint_product_key} grid."
                )

                if footprint_cfg.get("on_grid_mismatch", "warn") == "warn":
                    print(f"WARNING {pair_id}: footprint skipped; {footprint_warning}")

    raw_stats_after = raster_stats(array_after_footprint)

    display_array, norm_label, norm_offset = apply_display_normalization(
        array=array_after_footprint,
        product_key=product_key,
        config=config,
    )

    display_stats = {
        "display_mean": float(np.nanmean(display_array)),
        "display_median": float(np.nanmedian(display_array)),
        "display_std": float(np.nanstd(display_array)),
    }

    return {
        "pair_id": pair_id,
        "product_key": product_key,
        "product_role": product_role if product_role is not None else footprint_role,
        "variant": resolved_variant,
        "path": path,
        "display_array": display_array,
        "metadata": metadata,
        "raw_stats_before_footprint": raw_stats_before,
        "raw_stats_after_footprint": raw_stats_after,
        "display_stats": display_stats,
        "norm_label": norm_label,
        "norm_offset": norm_offset,
        "footprint_applied": footprint_applied,
        "footprint_product_key": footprint_product_key,
        "footprint_warning": footprint_warning,
    }


def item_to_summary_row(
    item: dict[str, object],
    dt_days=None,
) -> dict[str, object]:
    before = item["raw_stats_before_footprint"]
    after = item["raw_stats_after_footprint"]
    display = item["display_stats"]

    return {
        "pair_id": item["pair_id"],
        "dt_days": dt_days,
        "product_role": item["product_role"],
        "product_key": item["product_key"],
        "variant": item["variant"],
        "path": str(item["path"]),
        "footprint_applied": item["footprint_applied"],
        "footprint_product_key": item["footprint_product_key"],
        "footprint_warning": item["footprint_warning"],
        "count_before_footprint": int(before["count"]),
        "count_after_footprint": int(after["count"]),
        "count_removed_by_footprint": int(before["count"]) - int(after["count"]),
        "norm_label": item["norm_label"],
        "norm_offset": item["norm_offset"],
        "mean": after["mean"],
        "median": after["median"],
        "std": after["std"],
        "min": after["min"],
        "max": after["max"],
        "p02": after["p2"],
        "p05": after["p5"],
        "p50": after["p50"],
        "p95": after["p95"],
        "p98": after["p98"],
        "display_mean": display["display_mean"],
        "display_median": display["display_median"],
        "display_std": display["display_std"],
    }


def summary_table_from_items(
    items: list[dict[str, object]],
    dt_days_by_pair: Mapping[str, object] | None = None,
) -> pd.DataFrame:
    rows = []

    for item in items:
        pair_id = item["pair_id"]
        dt_days = None

        if dt_days_by_pair is not None:
            dt_days = dt_days_by_pair.get(pair_id, None)

        rows.append(item_to_summary_row(item, dt_days=dt_days))

    return pd.DataFrame(rows)


def panel_title(item: dict[str, object], dt_days=None) -> str:
    if dt_days is None or pd.isna(dt_days):
        return f"{item['pair_id']}\n{item['variant']} | {item['product_key']}"

    return (
        f"{item['pair_id']}\n"
        f"Δt = {dt_days} d | {item['variant']} | {item['product_key']}"
    )


def footnote_from_summary(
    summary_table: pd.DataFrame,
    shared_limits: tuple[float, float] | None,
    context: str,
    config: dict,
) -> str:
    lines = [context]

    if shared_limits is not None:
        lines.append(
            f"Shared display limits: {shared_limits[0]:.4g} to {shared_limits[1]:.4g}."
        )

    lines.append(
        "Display normalization and footprint masking are for visualization only; raw rasters are not modified."
    )

    for _, row in summary_table.iterrows():
        lines.append(
            f"{row['pair_id']}: {row['norm_label']}; "
            f"mean={row['mean']:.5g}, median={row['median']:.5g}; "
            f"footprint={row['footprint_applied']} ({row['footprint_product_key']}); "
            f"n={int(row['count_after_footprint'])}."
        )

    return "\n".join(lines)


def apply_layout(fig, footnote: str | None, config: dict, top: float = 0.91):
    if footnote:
        fig.text(
            0.01,
            0.01,
            footnote,
            ha="left",
            va="bottom",
            fontsize=config["plotting"].get("footnote_fontsize", 8),
        )
        fig.tight_layout(rect=[0, 0.18, 1, top])
    else:
        fig.tight_layout(rect=[0, 0, 1, top])


def save_table(summary_table: pd.DataFrame, path: Path):
    ensure_dir(path.parent)
    summary_table.to_csv(path, index=False)


def plot_master_group_role(
    master_group: pd.DataFrame,
    product_role: str,
    config: dict,
    variant: str = "auto",
    show: bool | None = None,
    save: bool | None = None,
    decimation: int | None = None,
):
    plotting = config["plotting"]

    if show is None:
        show = plotting.get("show_plots", True)

    if save is None:
        save = plotting.get("save_plots", False)

    if decimation is None:
        decimation = plotting.get("plot_decimation", 1)

    group = master_group.sort_values(["slave_date", "pair_id"]).copy()
    master = str(group["master"].iloc[0])

    items = []
    dt_days_by_pair = {}

    for _, row in group.iterrows():
        pair_id = row["pair_id"]
        dt_days_by_pair[pair_id] = row.get("dt_days_from_pair_id", None)

        item = read_product_for_plot(
            pair_id=pair_id,
            config=config,
            product_role=product_role,
            row=row,
            variant=variant,
        )

        items.append(item)

    summary_table = summary_table_from_items(
        items,
        dt_days_by_pair=dt_days_by_pair,
    )

    shared_limits = shared_limits_for_items(items, config=config)

    n = len(items)
    figsize_per_panel = plotting.get("figsize_per_panel", (7, 5.5))

    fig, axes = plt.subplots(
        1,
        n,
        figsize=(figsize_per_panel[0] * n, figsize_per_panel[1]),
        squeeze=False,
    )
    axes = axes.ravel()

    for ax, item in zip(axes, items):
        dt_days = dt_days_by_pair.get(item["pair_id"], None)

        spec = product_plot_spec(
            product_key=item["product_key"],
            array=item["display_array"],
            config=config,
            shared_limits=shared_limits,
        )

        plot_array = decimate_for_plot(item["display_array"], decimation=decimation)
        plot_array = np.ma.masked_invalid(plot_array)

        im = ax.imshow(
            plot_array,
            extent=item["metadata"].extent,
            cmap=spec["cmap"],
            vmin=spec["vmin"],
            vmax=spec["vmax"],
        )

        cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.03)
        cbar.set_label(spec["label"])

        ax.set_title(panel_title(item, dt_days=dt_days))
        format_axis(ax)

    footnote = None
    if plotting.get("title_detail_mode", "footnote") == "footnote":
        footnote = footnote_from_summary(
            summary_table=summary_table,
            shared_limits=shared_limits,
            context=f"Master {master}; role={product_role}; variant mode={variant}.",
            config=config,
        )

    fig.suptitle(f"Master {master} — {product_role} comparison", y=0.98)
    apply_layout(fig, footnote, config)

    if save:
        out_dir = ensure_dir(
            Path(config["paths"]["output_root"])
            / config["outputs"]["master_groups_subdir"]
            / master
        )
        out_png = out_dir / f"{master}_role_{product_role}_{variant}.png"
        out_csv = out_dir / f"{master}_role_{product_role}_{variant}_summary.csv"

        fig.savefig(out_png, dpi=plotting.get("dpi", 300), bbox_inches="tight")
        save_table(summary_table, out_csv)

    if show:
        plt.show()
    else:
        plt.close(fig)

    return fig, axes, summary_table


def plot_pair_roles(
    pair_id: str,
    product_roles: Iterable[str],
    config: dict,
    row: pd.Series | Mapping[str, object] | None = None,
    variant: str = "auto",
    show: bool | None = None,
    save: bool | None = None,
    decimation: int | None = None,
):
    plotting = config["plotting"]

    if show is None:
        show = plotting.get("show_plots", True)

    if save is None:
        save = plotting.get("save_plots", False)

    if decimation is None:
        decimation = plotting.get("plot_decimation", 1)

    product_roles = list(product_roles)

    items = [
        read_product_for_plot(
            pair_id=pair_id,
            config=config,
            product_role=role,
            row=row,
            variant=variant,
        )
        for role in product_roles
    ]

    dt_days = None
    if row is not None and "dt_days_from_pair_id" in row:
        dt_days = row["dt_days_from_pair_id"]

    summary_table = summary_table_from_items(
        items,
        dt_days_by_pair={pair_id: dt_days},
    )

    n = len(items)
    figsize_per_panel = plotting.get("figsize_per_panel", (7, 5.5))

    fig, axes = plt.subplots(
        1,
        n,
        figsize=(figsize_per_panel[0] * n, figsize_per_panel[1]),
        squeeze=False,
    )
    axes = axes.ravel()

    for ax, item in zip(axes, items):
        spec = product_plot_spec(
            product_key=item["product_key"],
            array=item["display_array"],
            config=config,
        )

        plot_array = decimate_for_plot(item["display_array"], decimation=decimation)
        plot_array = np.ma.masked_invalid(plot_array)

        im = ax.imshow(
            plot_array,
            extent=item["metadata"].extent,
            cmap=spec["cmap"],
            vmin=spec["vmin"],
            vmax=spec["vmax"],
        )

        cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.03)
        cbar.set_label(spec["label"])

        ax.set_title(panel_title(item, dt_days=dt_days))
        format_axis(ax)

    footnote = None
    if plotting.get("title_detail_mode", "footnote") == "footnote":
        footnote = footnote_from_summary(
            summary_table=summary_table,
            shared_limits=None,
            context=f"Pair {pair_id}; roles={', '.join(product_roles)}.",
            config=config,
        )

    fig.suptitle(f"{pair_id} — role quicklook", y=0.98)
    apply_layout(fig, footnote, config)

    if save:
        out_dir = ensure_dir(
            Path(config["paths"]["output_root"])
            / config["outputs"]["quicklooks_subdir"]
            / pair_id
        )
        role_text = "_".join(product_roles)
        out_png = out_dir / f"{pair_id}_roles_{role_text}_{variant}.png"
        out_csv = out_dir / f"{pair_id}_roles_{role_text}_{variant}_summary.csv"

        fig.savefig(out_png, dpi=plotting.get("dpi", 300), bbox_inches="tight")
        save_table(summary_table, out_csv)

    if show:
        plt.show()
    else:
        plt.close(fig)

    return fig, axes, summary_table


def plot_master_group_product(
    master_group: pd.DataFrame,
    product_key: str,
    config: dict,
    show: bool | None = None,
    save: bool | None = None,
    decimation: int | None = None,
):
    plotting = config["plotting"]

    if show is None:
        show = plotting.get("show_plots", True)

    if save is None:
        save = plotting.get("save_plots", False)

    if decimation is None:
        decimation = plotting.get("plot_decimation", 1)

    group = master_group.sort_values(["slave_date", "pair_id"]).copy()
    master = str(group["master"].iloc[0])

    items = []
    dt_days_by_pair = {}

    for _, row in group.iterrows():
        pair_id = row["pair_id"]
        dt_days_by_pair[pair_id] = row.get("dt_days_from_pair_id", None)

        item = read_product_for_plot(
            pair_id=pair_id,
            config=config,
            product_key=product_key,
            variant="explicit_product_key",
        )

        items.append(item)

    summary_table = summary_table_from_items(
        items,
        dt_days_by_pair=dt_days_by_pair,
    )

    shared_limits = shared_limits_for_items(items, config=config)

    n = len(items)
    figsize_per_panel = plotting.get("figsize_per_panel", (7, 5.5))

    fig, axes = plt.subplots(
        1,
        n,
        figsize=(figsize_per_panel[0] * n, figsize_per_panel[1]),
        squeeze=False,
    )
    axes = axes.ravel()

    for ax, item in zip(axes, items):
        dt_days = dt_days_by_pair.get(item["pair_id"], None)

        spec = product_plot_spec(
            product_key=item["product_key"],
            array=item["display_array"],
            config=config,
            shared_limits=shared_limits,
        )

        plot_array = decimate_for_plot(item["display_array"], decimation=decimation)
        plot_array = np.ma.masked_invalid(plot_array)

        im = ax.imshow(
            plot_array,
            extent=item["metadata"].extent,
            cmap=spec["cmap"],
            vmin=spec["vmin"],
            vmax=spec["vmax"],
        )

        cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.03)
        cbar.set_label(spec["label"])

        ax.set_title(panel_title(item, dt_days=dt_days))
        format_axis(ax)

    footnote = None
    if plotting.get("title_detail_mode", "footnote") == "footnote":
        footnote = footnote_from_summary(
            summary_table=summary_table,
            shared_limits=shared_limits,
            context=f"Master {master}; explicit product={product_key}.",
            config=config,
        )

    fig.suptitle(f"Master {master} — {product_key} comparison", y=0.98)
    apply_layout(fig, footnote, config)

    if save:
        out_dir = ensure_dir(
            Path(config["paths"]["output_root"])
            / config["outputs"]["master_groups_subdir"]
            / master
        )
        out_png = out_dir / f"{master}_product_{product_key}.png"
        out_csv = out_dir / f"{master}_product_{product_key}_summary.csv"

        fig.savefig(out_png, dpi=plotting.get("dpi", 300), bbox_inches="tight")
        save_table(summary_table, out_csv)

    if show:
        plt.show()
    else:
        plt.close(fig)

    return fig, axes, summary_table

In [ ]:
import importlib
import insar_sci.plotting as plotting
importlib.reload(plotting)

from insar_sci.plotting import plot_pair_roles, plot_master_group_role
from insar_sci.manifest import group_by_master

row0 = dev_manifest.iloc[0]

fig, axes, table_pair = plot_pair_roles(
    pair_id=row0["pair_id"],
    product_roles=["disp", "coh"],
    config=CONFIG,
    row=row0,
    variant="auto",
    show=True,
    save=False,
)

display(table_pair)

groups = group_by_master(dev_manifest)

fig, axes, table_master = plot_master_group_role(
    master_group=groups["20240110"],
    product_role="disp",
    config=CONFIG,
    variant="auto",
    show=True,
    save=False,
)

display(table_master)